In [12]:
import inspect
from src.models.deepmodels import SiameseDino

print(inspect.getmro(SiameseDino))

(<class 'src.models.deepmodels.SiameseDino'>, <class 'src.models.base.DeepModel'>, <class 'src.models.base.BaseModel'>, <class 'abc.ABC'>, <class 'torch.nn.modules.module.Module'>, <class 'object'>)


In [19]:
import pyRAPL
import time

def collect(a: int):
    for i in range(100000000):
        a += i

m = pyRAPL.Measurement("inference")
m.begin()
collect(1)
m.end()
print(m.result.pkg[0]*1e-6)

128.97126599999999


In [16]:
m.result

Result(label='inference', timestamp=1763061580.9247704, duration=1000221.122, pkg=[56506691.0], dram=None)

In [ ]:
import cv2
from typing import List
from tqdm import tqdm
from src.eval import Metric, Recall, Score
import numpy as np
import torch
from torch.utils.data import DataLoader
from PIL import Image


class OrbAlgorithm:

    def __init__(self, n_features: int):
        self.orb = cv2.ORB_create(nfeatures=n_features)
        self.bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

    def evaluate(self, gallery_dataloader: DataLoader, query_dataloader: DataLoader, metric: Metric) -> Score:
        self.distance_matrix = self._compute_distance_matrix(gallery_dataloader, query_dataloader)
        return metric.compute(self.distance_matrix, torch.tensor(query_dataloader.dataset.labels), torch.tensor(gallery_dataloader.dataset.labels))

    def _compute_distance_matrix(self,
                                   gallery_dataloader: DataLoader,
                                   query_dataloader: DataLoader
                                   ):
        query_paths = query_dataloader.dataset.paths
        gallery_paths = gallery_dataloader.dataset.paths
        descs_q = self._compute_img_descriptors_from_paths(query_paths)
        descs_g = self._compute_img_descriptors_from_paths(gallery_paths)
        n_q = len(descs_q)
        n_g = len(descs_g)
        dists_matrix = np.full((n_q, n_g), float('inf'), dtype=np.float32)
        print(f"Computing {n_q}x{n_g} distance matrix...")

        for i in tqdm(range(n_q)):
            for j in range(n_g):
                dists_matrix[i, j] = self._compute_average_distance(descs_q[i], descs_g[j])
    
        return torch.from_numpy(dists_matrix)
    
    def _compute_img_descriptors_from_paths(self, paths: List[str]):
        imgs_descriptors = []
        for path in paths:
            img_descriptor = self._get_img_descriptors(path)
            imgs_descriptors.append(img_descriptor)

        return imgs_descriptors
    
    def _get_img_descriptors(self, path_to_img: str):
        img = cv2.imread(path_to_img, cv2.IMREAD_GRAYSCALE)
        _, descriptors = self.orb.detectAndCompute(img, None)

        return descriptors

    def _compute_average_distance(self, des1: str, des2: str) -> float:
        if des1 is None or des2 is None:
            return float('inf')
        
        matches = self.bf.match(des1, des2)

        if not matches:
            return float('inf')

        avg_distance = sum(m.distance for m in matches) / len(matches)

        return avg_distance
    
    def get_nearest_neighbor(self, ref_path: str) -> str:
        highest_similarity = 0
        idx_closest = 0
        ref_descriptors = self._get_img_descriptors(ref_path)
        for idx, compared_descriptor in enumerate(self.image_descriptors):
            similarity = self._compute_average_distance(ref_descriptors, compared_descriptor)
            if similarity > highest_similarity:
                highest_similarity = similarity
                idx_closest = idx
        
        return self.paths[idx_closest]

In [1]:
from pathlib import Path
from src.data import DataPreparator, LazyLoadCollection, make_transform
from torch.utils.data import DataLoader

EXPERIMENT_DATA_PATH = Path("../data/augmented_data16")
ORIGINAL_DATA_PATH = Path("../data/small_data")

data_preparator = DataPreparator(ORIGINAL_DATA_PATH, EXPERIMENT_DATA_PATH, 42)

training_size = 0.7
data_splits = data_preparator.train_val_test_split(training_size, 0.2, 1)
train_paths, train_labels = data_splits["train"]
gallery_paths, gallery_labels = data_splits["gallery"]
val_query_paths, val_query_labels = data_splits["val_query"]
test_query_paths, test_query_labels = data_splits["test_query"]

Found 18 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:     192 samples from 12 classes (Augmented)
Gallery Loader:       67 samples from 18 classes (Original)
Val Query Loader:      3 samples from 3 classes (Original)
Test Query Loader:     3 samples from 3 classes (Original)


In [2]:
RESIZE_SIZE = 224

gallery_dataset = LazyLoadCollection(gallery_paths, gallery_labels, transform=make_transform(RESIZE_SIZE))
gallery_dataloader = DataLoader(gallery_dataset, batch_size=32)

val_dataset = LazyLoadCollection(val_query_paths, val_query_labels, transform=make_transform(RESIZE_SIZE))
val_dataloader = DataLoader(val_dataset, batch_size=32)

In [ ]:
from src.models.cvmodels import FeatureBasedEstimator
from src.models.feature_extractors import RapidTextExtractor, OrbFeatureExtractor
from src.models.deepmodels import SiameseDino
from transformers import AutoImageProcessor, AutoModel
from hydra import initialize, compose
import torch

with initialize(config_path="../config", version_base=None):
    cfg = compose(config_name="base_config")

cfg.base.device = 'cpu'
checkpoint = torch.load("../model_checkpoints/sandy-fire-218.pth")

backbone_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(backbone_name)
backbone = AutoModel.from_pretrained(backbone_name, dtype=torch.float32)

model2 = SiameseDino(backbone, processor, cfg)
model2.load_state_dict(checkpoint)

#rapidocr = RapidTextExtractor()
orb = OrbFeatureExtractor(1000)
feature_extractors = [orb]
weights = [1]
model1 = FeatureBasedEstimator(feature_extractors, weights)
""" feature_extractors = [orb, rapidocr]
weights = [0.5, 0.5]
model2 = FeatureBasedEstimator(feature_extractors, weights)
weights = [0.7, 0.3]
model3 = FeatureBasedEstimator(feature_extractors, weights) """

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

' feature_extractors = [orb, rapidocr]\nweights = [0.5, 0.5]\nmodel2 = FeatureBasedEstimator(feature_extractors, weights)\nweights = [0.7, 0.3]\nmodel3 = FeatureBasedEstimator(feature_extractors, weights) '

In [2]:
from src.utils import ModelReport
from src.eval import Recall

metric = Recall()
modelreport = ModelReport({"siamesedino": model2, "orb": model1})#, "0.5_orb+0.5_rapidocr": model2, "0.7_orb+0.3_rapidocr": model3})
df = modelreport.generate_report("../data/original_data_color_invariant", "random", metric)

Found 88 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:      16 samples from 1 classes (Augmented)
Gallery Loader:      265 samples from 88 classes (Original)
Val Query Loader:     87 samples from 87 classes (Original)
Test Query Loader:     0 samples from 0 classes (Original)
['../data/original_data_color_invariant/003/IMG_20240220_225223445.jpg', '../data/original_data_color_invariant/003/IMG_20240220_175326581.jpg', '../data/original_data_color_invariant/003/IMG_20240220_170923133.jpg', '../data/original_data_color_invariant/015/IMG_20240220_220808999.jpg', '../data/original_data_color_invariant/015/IMG_20240220_171138085.jpg', '../data/original_data_color_invariant/015/IMG_20240220_175042472.jpg', '../data/original_data_color_invariant/030/IMG_20240220_224722504.jpg', '../data/original_data_color_invariant/030/IMG_20240220_174206670.jpg', '../data/original_data_color_invariant/030/I

Computing features: 100%|██████████| 264/264 [00:25<00:00, 10.21it/s]


✅ Gallery prepared: 264 images
Computing queries features...


Computing features: 100%|██████████| 88/88 [00:08<00:00, 10.29it/s]


Elapsed time = 92.69s
Elapsed time = 0.09s
Elapsed time = 0.73s


In [3]:
df

,evaluation time,inference time,recall@1,recall@3,recall@10
siamesedino,2.651978,0.173975,0.579545,0.727273,0.840909
orb,92.692321,0.732617,0.829545,0.863636,0.886364


In [1]:
from pathlib import Path
from src.utils import Browser

ORIGINAL_DATA_PATH = Path("../data/sorted_original_good_orientation")
browser = Browser(ORIGINAL_DATA_PATH)
splits = browser.sample_leave_k_out(1)

1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


In [2]:
splits

([], [], [], [])